#### Importing Dependencies.
Make sure to create an env using <b> python 3.10 </b> to prevent facing any library issue.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
from tqdm import tqdm
import hazm

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

from torch.utils.data import DataLoader, Dataset 

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"This device supports '{device}'")

_____________________________________________
#### Loading Chunked Wikipedia Farsi Dataset

In [ ]:
df = pd.read_csv("chunk_wiki_dataset.csv")
chunks = df['chunk']
chunks = chunks.dropna().astype(str).tolist()
print(f"Number of total chunks: {len(chunks)}")
print(f"Number of tokens in each chunk is about: 512")
print(f"Total number of tokens: {len(chunks)*512}")

_______________________________
#### Loading Embedding Model
* Link: https://huggingface.co/heydariAI/persian-embeddings

In [ ]:
embedding_model_path = os.path.join("C:/Users/Amir/Desktop/Models/Embeddings/Persian Embedding")

# Loading model as a high-level helper
# from transformers import pipeline
# pipe = pipeline("feature-extraction", model="heydariAI/persian-embeddings", cache_dir=embedding_model_path)

# Loading model directly
from transformers import AutoTokenizer, AutoModel
embedding_tokenizer = AutoTokenizer.from_pretrained("heydariAI/persian-embeddings", cache_dir=embedding_model_path)
embedding_model = AutoModel.from_pretrained("heydariAI/persian-embeddings", cache_dir=embedding_model_path)
embedding_model = embedding_model.to(device)
embedding_model.eval()

print("Embedding Model has been loaded successfully.")

_________________________________
#### Chunk Text Into Embeddings

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] # First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# === Defining a function to convert data into embeddings === #
def chunks_to_embeddings(chunks, embedding_tokenizer, embedding_model, batch_size=128):
    
    all_embeddings = []
    for i in tqdm(range(0, len(chunks), batch_size)):
        batch = chunks[i:i + batch_size]

        encoded_input = embedding_tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=512)

        encoded_input = {k: v.to(device) for k, v in encoded_input.items()}

        # Gradient will not be calculated wich will result in using less RAM, VRAM and more calculation speed
        # torch.inference_mode() ~ torch.no_grad()
        with torch.inference_mode():
            model_output = embedding_model(**encoded_input)

        embeddings = mean_pooling(model_output, encoded_input["attention_mask"])

        all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings)

In [ ]:
# Change the flag to 'True' to start converting texts into embeddings
training_loop_flag = False

if training_loop_flag:
    embeddings = chunks_to_embeddings(chunks, embedding_tokenizer, embedding_model, batch_size=256)
    np.save("farsi_wikipedia_embeddings.npy", embeddings)
else:
    embeddings = np.load("wiki_embeddings.npy")
    
print(f"Embeddings Shape: {embeddings.shape}")

_____________________________________________________
#### Saving Embeddings and Texts in Vector DataBase
* For vector db we're gonna use 'qdrant'

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(url=f"http://localhost:6333")
client.create_collection(
    collection_name="wiki_farsi",
    vectors_config=VectorParams(
        size=embeddings[0].shape[0], 
        distance=Distance.COSINE)
)

In [ ]:
from qdrant_client.models import PointStruct

# Fortunately the arrangement of texts and embeddings are same
# embeddings[n] is corrseponded to chunks[n]

points = []
for i in tqdm(range(len(chunks))):
    points.append(
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload={"text": chunks[i]
            }
        )
    )

In [ ]:
BATCH_SIZE = 512

for i in tqdm(range(0, len(points), BATCH_SIZE)):
    client.upsert(
        collection_name="wiki_farsi",
        points=points[i:i+BATCH_SIZE]
    )

    print(f"Inserted {i}")